# MathScribe: Colab runner

This notebook is a **thin runner**. It has no training logic. It only:

1. mounts Google Drive (optional),
2. clones the repo,
3. runs `scripts/setup_env.sh --colab`,
4. logs in to the HF Hub,
5. runs the same CLI command that you run locally.

Before you start:
- Select a GPU runtime (*Runtime → Change runtime type*).
- Add the secret `HF_TOKEN` (a token with **write** access) in the Colab *Secrets* panel.

**Resume after a disconnect:** set `RESUME = True` and run all cells again.
The trainer continues from the most advanced checkpoint: the Hub `last-checkpoint/` folder, or the local `output_dir`.

In [ ]:
# ---- Run settings: edit these ----
REPO_URL = "https://github.com/Djalal-H/FineTuning.git"
BRANCH = "main"
CONFIG = "configs/train/sft_smoke.yaml"
HUB_REPO = ""  # e.g. "<hf-user>/mathscribe-sft-smoke"; empty = no Hub push
RESUME = False  # True = continue from the latest checkpoint
USE_DRIVE = False  # True = keep output_dir on Google Drive
EXTRA_OVERRIDES = []  # e.g. ["train.max_steps=30"]

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os

WORKDIR = "/content/mathscribe"
OUTPUT_ROOT = "/content/outputs"
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/mathscribe/outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

if os.path.isdir(f"{WORKDIR}/.git"):
    !git -C {WORKDIR} fetch origin {BRANCH}
    !git -C {WORKDIR} checkout {BRANCH}
    !git -C {WORKDIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

In [ ]:
!bash scripts/setup_env.sh --colab

In [ ]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!hf auth whoami

In [ ]:
import shlex
from pathlib import Path

run_name = Path(CONFIG).stem
overrides = [f"train.output_dir={OUTPUT_ROOT}/{run_name}", *EXTRA_OVERRIDES]
if HUB_REPO:
    overrides += ["hub.push=true", f"hub.repo_id={HUB_REPO}"]

cmd = ["python", "-m", "mathscribe.train.sft", "--config", CONFIG]
for o in overrides:
    cmd += ["--set", o]
if RESUME:
    cmd.append("--resume")

CMD = shlex.join(cmd)
print(CMD)

In [ ]:
!{CMD}

In [ ]:
import json

with open(f"{OUTPUT_ROOT}/{run_name}/run_info.json") as f:
    info = json.load(f)
print(json.dumps(info, indent=2)[:3000])